# 04b · Basamak direnci (modelden bağımsız, eğitim yok)
Ham 10 Hz kayıttaki her akım basamağı için **R(h) = ΔV/ΔI** değerini 0.1 / 1 / 3 / 10 s ufuklarında ölçer
(|ΔI| ≥ 1 A; basamaktan önce ≥ 1 s ve sonra h boyunca sabit akım). Sıcaklık başına medyan, IQR ve %95 GA;
SoC / yön / |ΔI| kırılımları; Arrhenius uyumu (tümü ve −20 hariç).

Öğrenilen R(T) ile karşılaştırma: `results/checkpoints` içindeki RC/ohmik katmanlı modellerin birim basamak yanıtı
(ana model C, `final_c`, şekil (b)). **C için kıyas ufku 3 s** (3 s ohmik yumuşatma). 10 s ufku önceki akımların yavaş
kolu tam sönmediği için %3–7 düşük çıkabilir → alt sınır olarak okunmalı.

Çıktılar: `results/excel/04b_basamak_direnci.xlsx`, `results/series/04b_basamak_olaylari.csv.gz`,
`graphs/04b_step_resistance.png/.pdf`. Süre: birkaç dakika (GPU kullanmaz; 03c eğitimi sürerken de çalışabilir).

Çalıştırma: hücreleri sırayla çalıştırın (**Run All**).

In [ ]:
import os; ROOT = os.path.abspath('..')  # repository root (run notebooks from the notebooks/ folder)

import sys, re, json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

ROOT = Path(ROOT)
sys.path.insert(0, str(ROOT))

CFG = dict(
    rate=10,                 # Hz, ham kayıt
    horizons=(0.1, 1.0, 3.0, 10.0),   # s
    dI_min=1.0,              # A, en küçük basamak
    tol_abs=0.10,            # A, "sabit akım" toleransı
    tol_rel=0.05,            # |ΔI| oranı; tol = max(tol_abs, tol_rel·|ΔI|)
    pre_min_s=1.0,           # basamak öncesi en az bu kadar sabit akım
    pre_cap_s=30.0,          # öncesi sabitlik sayımı üst sınırı
    settled_s=5.0,           # "oturmuş" alt küme: öncesi ≥ bu kadar sabit (önceki geçici etkiler sönük)
    gap_s=0.5,               # bundan büyük zaman aralığı = kayıt kesintisi
    t_tol_s=0.06,            # hedef zamana en yakın örnek toleransı
    r_max=1.0,               # Ω, fiziksel olmayan değerleri at
    soc_bins=(0.0, 0.2, 0.4, 0.6, 0.8, 1.01),
    dI_bins=(1.0, 2.0, 4.0, 8.0, 100.0),
    nominal_ah=2.9,
)
K_B = 8.617333e-5  # eV/K



## Fonksiyonlar

In [ ]:
# ───────────────────────────── olay çıkarımı ─────────────────────────────
def find_events(t, V, I, Ah, Tb, cfg=CFG):
    """Tek dosyadan basamak olayları. Dönüş: DataFrame (her satır bir basamak)."""
    n = len(t)
    rate, H = cfg['rate'], cfg['horizons']
    seg = np.cumsum(np.concatenate([[1], np.diff(t) > cfg['gap_s']]))
    n_post0 = rate                                    # I_post için ilk ~1 s
    pre_min = max(int(round(cfg['pre_min_s'] * rate)) - 1, 1)
    pre_cap = int(round(cfg['pre_cap_s'] * rate))
    dI1 = np.abs(np.diff(I, prepend=I[0]))
    cand = np.where(dI1 >= 0.3 * cfg['dI_min'])[0]
    rows = []
    for k in cand:
        if k < 1 or k + n_post0 >= n or seg[k] != seg[k - 1] or seg[k + n_post0 - 1] != seg[k]:
            continue
        i_pre = I[k - 1]
        post = I[k + 1:k + n_post0] if n_post0 > 1 else I[k:k + 1]
        i_post = float(np.median(post))
        dI = i_post - i_pre
        if abs(dI) < cfg['dI_min']:
            continue
        tol = max(cfg['tol_abs'], cfg['tol_rel'] * abs(dI))
        # basamak öncesi sabitlik
        m = 0
        while m < pre_cap and k - 2 - m >= 0 and seg[k - 2 - m] == seg[k] and abs(I[k - 2 - m] - i_pre) <= tol:
            m += 1
        if m < pre_min:
            continue
        ramp = abs(I[k] - i_post) > tol
        t0, v0 = t[k - 1], V[k - 1]
        row = {'k': int(k), 't': float(t0), 'I_pre': float(i_pre), 'I_post': i_post, 'dI': float(dI),
               'pre_steady_s': (m + 1) / rate, 'ramp': bool(ramp),
               'soc': float(np.clip((cfg['nominal_ah'] + Ah[k - 1]) / cfg['nominal_ah'], 0, 1)), 'T_batt': float(Tb[k - 1])}
        for h in H:
            j = int(np.searchsorted(t, t0 + h - cfg['t_tol_s']))
            r = np.nan
            if j < n and abs(t[j] - (t0 + h)) <= cfg['t_tol_s'] and seg[j] == seg[k] and j >= k:
                lo = k + 1 if ramp else k
                if not (ramp and j == k) and (j < lo or np.all(np.abs(I[lo:j + 1] - i_post) <= tol)):
                    r = (V[j] - v0) / dI
                    if not (0 < r <= cfg['r_max']):
                        r = np.nan
            row[f'R_{h:g}s'] = r
        rows.append(row)
    return pd.DataFrame(rows)


# ───────────────────────────── özet istatistikler ─────────────────────────────
def med_ci(x):
    """Medyan ve sıralama istatistiğine dayalı %95 güven aralığı (dağılımdan bağımsız)."""
    x = np.sort(np.asarray(x)[np.isfinite(x)])
    n = len(x)
    if n == 0:
        return np.nan, np.nan, np.nan
    lo = int(np.floor(n / 2 - 1.96 * np.sqrt(n) / 2)); hi = int(np.ceil(n / 2 + 1.96 * np.sqrt(n) / 2))
    return float(np.median(x)), float(x[max(lo, 0)]), float(x[min(hi, n - 1)])


def summarize(ev, by, hcols, scale=1e3):
    rows = []
    for key, g in ev.groupby(by, sort=False, observed=True):
        key = key if isinstance(key, tuple) else (key,)
        base = dict(zip(by, key))
        base['T_batt medyan (°C)'] = float(g['T_batt'].median())
        for h in hcols:
            x = g[h].values * scale
            md, lo, hi = med_ci(x)
            ok = np.isfinite(x)
            rows.append({**base, 'ufuk': h.replace('R_', ''), 'n': int(ok.sum()), 'medyan (mΩ)': md,
                         'CI95 alt': lo, 'CI95 üst': hi,
                         'Q25': float(np.nanpercentile(x, 25)) if ok.any() else np.nan,
                         'Q75': float(np.nanpercentile(x, 75)) if ok.any() else np.nan})
    out = pd.DataFrame(rows)
    if len(out):
        out['ufuk'] = pd.Categorical(out['ufuk'], [h.replace('R_', '') for h in hcols], ordered=True)
    return out


def temp_value(label, fallback):
    """Etiketten sıcaklık sayısı ('25degC', 'n10degC', '-20' …); bulunamazsa ölçülen medyan."""
    s = str(label)
    m = re.search(r'(n|neg|minus|-|−)?\s*(\d+)', s, re.I)
    if not m:
        return fallback
    v = float(m.group(2))
    return -v if m.group(1) else v


def arrhenius(file_tab, hcols, cold_label):
    """Dosya medyanları üzerinden ln R = ln A + Ea/(k_B·T). Tümüyle ve −20 hariç (dışa uzatma testi)."""
    rows = []
    for h in hcols:
        d = file_tab[['etiket', 'T_batt medyan (°C)', h]].dropna()
        d = d[d[h] > 0]
        for name, sub in [('tüm sıcaklıklar', d), ('−20 hariç (eğitim aralığı)', d[d.etiket != cold_label])]:
            if sub.etiket.nunique() < 3:
                continue
            x = 1.0 / (sub['T_batt medyan (°C)'].values + 273.15); y = np.log(sub[h].values)
            b, a = np.polyfit(x, y, 1)
            yhat = a + b * x
            r2 = 1 - np.sum((y - yhat) ** 2) / np.sum((y - y.mean()) ** 2)
            row = {'ufuk': h.replace('R_', ''), 'uyum': name, 'dosya': len(sub), 'Ea (eV)': b * K_B,
                   'Ea (kJ/mol)': b * K_B * 96.485, 'R²': r2}
            cold = d[d.etiket == cold_label]
            if len(cold):
                tc = cold['T_batt medyan (°C)'].median() + 273.15
                row['−20 tahmin (mΩ)'] = float(np.exp(a + b / tc))
                row['−20 ölçülen medyan (mΩ)'] = float(cold[h].median())
            rows.append(row)
    return pd.DataFrame(rows)


# ───────────────────────────── model karşılaştırması ─────────────────────────────
def model_step_R(sd, t_n, n_s, ohm_smooth=3):
    """Öğrenilmiş RC katmanının birim basamak yanıtı (Ω), n_s saniye (1 Hz örnek) sonra.
    η(n) = R0·min(n, s)/s + Σ R_k·(1 − a_k^n);  olay modülasyonu (det) nominal (m = 0) alınır."""
    g = lambda k: np.asarray(sd[k].detach().cpu().numpy() if hasattr(sd[k], 'detach') else sd[k], dtype=float)
    r0 = np.exp(g('rc.r0_a') + g('rc.r0_b') * (t_n - 1))
    out = r0 * min(n_s, ohm_smooth) / ohm_smooth
    if 'rc.log_tau' in sd:
        tau = np.clip(np.exp(g('rc.log_tau')), 1.0, 60.0)
        rk = np.exp(g('rc.rk_a') + g('rc.rk_b') * (t_n - 1))
        out = out + np.sum(rk * (1 - np.exp(-n_s / tau)))
    return float(out)


def model_compare(root, temps_c, horizons):
    try:
        import torch
    except ImportError:
        print('ℹ️  torch yok → model karşılaştırması atlandı'); return None, None
    nf = root / 'results' / 'configs' / 'normalizer.json'
    if not nf.exists():
        print(f'ℹ️  {nf} yok → model karşılaştırması atlandı'); return None, None
    st = json.load(open(nf))['temp']
    tn = lambda T: (T - st['min']) / (st['max'] - st['min'] + 1e-8)
    rows, prm = [], []
    for p in sorted((root / 'results' / 'checkpoints').glob('*/*.pt')):
        if p.parent.name.lower().startswith('tun'):      # ayarlama koşuları atlanır
            continue
        try:
            ck = torch.load(p, map_location='cpu', weights_only=False)
        except Exception as e:
            print(f'⚠️  {p.name}: {e}'); continue
        sd = ck
        if isinstance(ck, dict):
            for key in ('state_dict', 'model', 'state', 'model_state'):
                if key in ck and isinstance(ck[key], dict):
                    sd = ck[key]; break
        if not isinstance(sd, dict) or 'rc.r0_a' not in sd:
            continue
        # spec (ohmik yumuşatma süresi) ve okunur model adı: koşu JSON'u (results/runs/<aşama>/<aynı ad>.json)
        rec = {}
        js = root / 'results' / 'runs' / p.parent.name / (p.stem + '.json')
        if js.exists():
            try:
                rec = json.load(open(js, encoding='utf-8'))
            except Exception:
                rec = {}
        spec = rec.get('spec') or (ck.get('spec', {}) if isinstance(ck, dict) else {})
        s = int(spec.get('rc', {}).get('ohm_smooth', 3)) if isinstance(spec, dict) else 3
        s = max(s, 1)
        m = re.match(r'(.+)_s(\d+)$', p.stem)
        model, seed = (m.group(1), int(m.group(2))) if m else (p.stem, -1)
        model = rec.get('name', model)
        taus = np.clip(np.exp(sd['rc.log_tau'].numpy()), 1, 60) if 'rc.log_tau' in sd else []
        prm.append({'aşama': p.parent.name, 'model': model, 'seed': seed, 'ohm_smooth': s,
                    'τ (s)': ' / '.join(f'{x:.1f}' for x in taus)})
        for lab, T in temps_c.items():
            for h in horizons:
                n_s = max(int(round(h)), 1)
                if h < 1:
                    continue
                rows.append({'aşama': p.parent.name, 'model': model, 'seed': seed, 'etiket': lab, 'T (°C)': T,
                             'ufuk': f'{h:g}s', 'R_model (mΩ)': 1e3 * model_step_R(sd, tn(T), n_s, s)})
    if not rows:
        print('ℹ️  RC katmanlı checkpoint bulunamadı'); return None, None
    return pd.DataFrame(rows), pd.DataFrame(prm)


# ───────────────────────────── şekil ─────────────────────────────
def make_figure(summ, cmp_tab, out):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    plt.rcParams.update({'font.size': 9, 'font.family': 'DejaVu Sans'})
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.0), constrained_layout=True)
    ax = axes[0]
    cols = ['#1f4e79', '#2e86c1', '#e67e22', '#8e44ad']
    for c, (h, g) in zip(cols, summ.groupby('ufuk', sort=True, observed=True)):
        g = g.sort_values('T (°C)')
        ax.errorbar(g['T (°C)'], g['medyan (mΩ)'], yerr=[g['medyan (mΩ)'] - g['Q25'], g['Q75'] - g['medyan (mΩ)']],
                    marker='o', ms=4, lw=1.2, capsize=2, color=c, label=f'{h} after step')
    ax.set_yscale('log'); ax.set_xlabel('Temperature (°C)'); ax.set_ylabel('Step resistance ΔV/ΔI (mΩ)')
    ax.set_title('(a) Measured from data (median, IQR)', fontsize=9, loc='left')
    ax.legend(frameon=False, fontsize=7); ax.grid(alpha=.3, which='both')
    ax = axes[1]
    if cmp_tab is not None and len(cmp_tab):
        prev = None
        mean_at = lambda h: cmp_tab[cmp_tab.ufuk == h].groupby('T (°C)')['R_model (mΩ)'].mean().values
        flat = len(mean_at('3s')) and len(mean_at('10s')) and np.allclose(mean_at('3s'), mean_at('10s'), rtol=1e-6)
        for c, h in zip(cols[1:], ['1s', '3s', '10s']):
            d = summ[summ.ufuk == h].sort_values('T (°C)')
            mm = cmp_tab[cmp_tab.ufuk == h].groupby('T (°C)')['R_model (mΩ)'].agg(['mean', 'std']).reset_index()
            if not len(d) or not len(mm):
                continue
            ax.plot(d['T (°C)'], d['medyan (mΩ)'], 'o', color=c, ms=5, label=f'data, {h}')
            # yalnız ohmik kol (C): 3 s'den sonra yanıt sabit → aynı eğriyi tekrar çizme
            if prev is not None and np.allclose(mm['mean'].values, prev, rtol=1e-6):
                continue
            prev = mm['mean'].values
            ax.errorbar(mm['T (°C)'], mm['mean'], yerr=mm['std'], color=c, lw=1.2, ls='--', capsize=2,
                        label=f'learned, {h}' + (' and later' if h == '3s' and flat else ''))
        ax.set_yscale('log'); ax.legend(frameon=False, fontsize=6.5, ncol=1)
        ax.set_title('(b) Learned resistance vs. data', fontsize=9, loc='left')
    else:
        r = summ.assign(ufuk=summ.ufuk.astype(str)).pivot_table(index='T (°C)', columns='ufuk', values='medyan (mΩ)')
        r = r[[str(u) for u in summ.ufuk.cat.categories if str(u) in r.columns]]
        ref = r.loc[r.index.max()]
        for c, h in zip(cols, r.columns):
            ax.plot(r.index, r[h] / ref[h], 'o-', color=c, ms=4, label=f'{h} after step')
        ax.set_ylabel('Ratio to 25 °C'); ax.legend(frameon=False, fontsize=7)
        ax.set_title('(b) Increase relative to 25 °C', fontsize=9, loc='left')
    ax.set_xlabel('Temperature (°C)'); ax.grid(alpha=.3, which='both')
    if cmp_tab is not None and len(cmp_tab):
        ax.set_ylabel('Step resistance (mΩ)')
    out.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out.with_suffix('.png'), dpi=300); fig.savefig(out.with_suffix('.pdf'))
    plt.close(fig)


# ───────────────────────────── ana akış ─────────────────────────────
def main():
    sys.path.insert(0, str(ROOT))
    from data_utils.data import load_raw
    from data_utils.files import DATA_FOLDERS, ROLE_FILES, TEST_FILES
    t_start = time.time()
    files = [(t, r, 'eğitim' if r != 'NN' else 'val', fn) for t, rf in ROLE_FILES.items() for r, fn in rf.items()]
    files += [(t, c, 'test', fn) for t, c, fn in TEST_FILES]
    seen, evs = set(), []
    for lab, cyc, role, fn in files:
        fp = ROOT / 'data' / DATA_FOLDERS[lab] / fn
        if fp in seen or not fp.exists():
            if not fp.exists(): print(f'⚠️  yok: {fp}')
            continue
        seen.add(fp)
        raw = load_raw(fp)
        ev = find_events(raw['Time'], raw['Voltage'], raw['Current'], raw['Ah'], raw['Battery_Temp_degC'])
        if not len(ev):
            print(f'– {lab} {cyc}: olay yok'); continue
        ev.insert(0, 'dosya', fn); ev.insert(0, 'küme', role); ev.insert(0, 'çevrim', cyc); ev.insert(0, 'etiket', lab)
        evs.append(ev)
        hc = [c for c in ev.columns if c.startswith('R_')]
        print(f'✅ {lab:>8s} {str(cyc):10s} {len(ev):6d} basamak | ' +
              ' · '.join(f'{c[2:]} {1e3 * np.nanmedian(ev[c]):.1f} mΩ (n={ev[c].notna().sum()})' for c in hc))
    ev = pd.concat(evs, ignore_index=True)
    hcols = [c for c in ev.columns if c.startswith('R_')]
    # sıcaklık etiketi → sayısal (etiket ayrıştırılamazsa ölçülen medyan)
    tmed = ev.groupby('etiket')['T_batt'].median()
    tval = {lab: temp_value(lab, float(tmed[lab])) for lab in tmed.index}
    ev['T (°C)'] = ev['etiket'].map(tval)
    order = sorted(tval, key=lambda l: -tval[l])
    ev['etiket'] = pd.Categorical(ev['etiket'], order, ordered=True)
    cold = order[-1]; warm = order[0]
    ev['yön'] = np.where(ev['dI'] < 0, 'deşarj artışı (ΔI<0)', 'deşarj azalışı / şarj (ΔI>0)')
    ev['SoC aralığı'] = pd.cut(ev['soc'], CFG['soc_bins'], right=False)
    ev['|ΔI| aralığı (A)'] = pd.cut(ev['dI'].abs(), CFG['dI_bins'], right=False)

    by_T = ['etiket', 'T (°C)']
    summ = summarize(ev.sort_values('etiket'), by_T, hcols)
    summ['etiket'] = pd.Categorical(summ['etiket'], order, ordered=True)
    ref = summ[summ.etiket == warm].assign(ufuk=lambda d: d.ufuk.astype(str)).set_index('ufuk')['medyan (mΩ)']
    summ[f'oran / {warm}'] = summ['medyan (mΩ)'].values / summ['ufuk'].astype(str).map(ref).astype(float).values
    settled = summarize(ev[ev.pre_steady_s >= CFG['settled_s']].sort_values('etiket'), by_T, hcols)
    by_soc = summarize(ev.sort_values(['etiket', 'SoC aralığı']), by_T + ['SoC aralığı'], hcols)
    by_dir = summarize(ev.sort_values('etiket'), by_T + ['yön'], hcols)
    by_dI = summarize(ev.sort_values(['etiket', '|ΔI| aralığı (A)']), by_T + ['|ΔI| aralığı (A)'], hcols)
    for d in (by_soc, by_dI):
        for c in ('SoC aralığı', '|ΔI| aralığı (A)'):
            if c in d: d[c] = d[c].astype(str)
    ftab = ev.groupby(['etiket', 'çevrim', 'küme', 'dosya'], observed=True).agg(
        **{'T_batt medyan (°C)': ('T_batt', 'median'), 'basamak': ('dI', 'size')},
        **{h: (h, lambda x: 1e3 * np.nanmedian(x) if np.isfinite(x).any() else np.nan) for h in hcols}).reset_index()
    ftab['etiket'] = ftab['etiket'].astype(str)
    arr = arrhenius(ftab, hcols, cold)
    ftab_out = ftab.rename(columns={h: f'{h[2:]} medyan (mΩ)' for h in hcols})

    cmp_tab, prm = model_compare(ROOT, {lab: tval[lab] for lab in order}, CFG['horizons'])
    cmp_sum = None
    if cmp_tab is not None:
        g = cmp_tab.groupby(['aşama', 'model', 'etiket', 'T (°C)', 'ufuk'], sort=False)['R_model (mΩ)']
        cmp_sum = g.agg(['mean', 'std', 'count']).reset_index().rename(
            columns={'mean': 'model ort (mΩ)', 'std': 'model sd', 'count': 'seed'})
        dmed = summ.assign(ufuk=summ.ufuk.astype(str), etiket=summ.etiket.astype(str)).set_index(['etiket', 'ufuk'])['medyan (mΩ)']
        cmp_sum['veri medyan (mΩ)'] = [dmed.get((e, u), np.nan) for e, u in zip(cmp_sum.etiket, cmp_sum.ufuk)]
        cmp_sum['model / veri'] = cmp_sum['model ort (mΩ)'] / cmp_sum['veri medyan (mΩ)']
        hord = [f'{h:g}s' for h in CFG['horizons'] if h >= 1]
        cmp_sum['ufuk'] = pd.Categorical(cmp_sum['ufuk'], hord, ordered=True)
        cmp_sum['etiket'] = pd.Categorical(cmp_sum['etiket'], order, ordered=True)

    # kayıt
    xl = ROOT / 'results' / 'excel' / '04b_basamak_direnci.xlsx'; xl.parent.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(xl) as w:
        summ.to_excel(w, sheet_name='Özet', index=False)
        settled.to_excel(w, sheet_name=f'Oturmuş (öncesi ≥{CFG["settled_s"]:g} s)'[:31], index=False)
        by_soc.to_excel(w, sheet_name='SoC', index=False)
        by_dir.to_excel(w, sheet_name='Yön', index=False)
        by_dI.to_excel(w, sheet_name='|ΔI|', index=False)
        ftab_out.to_excel(w, sheet_name='Dosya bazında', index=False)
        arr.to_excel(w, sheet_name='Arrhenius', index=False)
        if cmp_sum is not None:
            cmp_sum.to_excel(w, sheet_name='Model karşılaştırma', index=False)
            prm.to_excel(w, sheet_name='Model checkpointleri', index=False)
            cmp_tab.to_excel(w, sheet_name='Model ham (seed)', index=False)
        pd.DataFrame([{'ayar': k, 'değer': str(v)} for k, v in CFG.items()]).to_excel(w, sheet_name='Ayarlar', index=False)
    ser = ROOT / 'results' / 'series' / '04b_basamak_olaylari.csv.gz'; ser.parent.mkdir(parents=True, exist_ok=True)
    ev.to_csv(ser, index=False, compression='gzip')
    gsel = cmp_tab
    if cmp_tab is not None:   # şekil: ana model C ('final_c'); yoksa diğer final aşaması; yoksa ilk grup
        a = cmp_tab['aşama'].str.lower()
        cand = cmp_tab[a == 'final_c']
        if not len(cand):
            cand = cmp_tab[a.str.contains('final') & ~a.str.contains('small')]
        cand = cand if len(cand) else cmp_tab
        first = cand.groupby(['aşama', 'model']).size().index[0]
        gsel = cand[(cand['aşama'] == first[0]) & (cand['model'] == first[1])]
        print(f'Şekil (b) için model: {first[0]}/{first[1]}')
    make_figure(summ, gsel, ROOT / 'graphs' / '04b_step_resistance')

    # konsol özeti
    print('\n══ Basamak direnci (medyan, mΩ) ══')
    print(summ.pivot_table(index=['etiket', 'T (°C)'], columns='ufuk', values='medyan (mΩ)', observed=True)
          .round(1).to_string())
    print(f'\nOran / {warm}:')
    print(summ.pivot_table(index='etiket', columns='ufuk', values=f'oran / {warm}', observed=True).round(2).to_string())
    print('\nArrhenius:'); print(arr.round(3).to_string(index=False))
    if cmp_sum is not None:
        print('\nModel / veri (seed ortalaması):')
        print(cmp_sum.pivot_table(index=['aşama', 'model', 'etiket'], columns='ufuk', values='model / veri', observed=True)
              .round(2).to_string())
    print(f'\n→ {xl}\n→ {ser}\n→ graphs/04b_step_resistance.png/.pdf   ({time.time() - t_start:.0f} s)')


## Çalıştır

In [ ]:
main()

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(ROOT / 'graphs' / '04b_step_resistance.png')))